# Módulo 04 — Visualização Científica de Dados: `Matplotlib` e `Seaborn`

## 1. Por que Visualizar Antes de Modelar?

Na engenharia, **olhar apenas para médias e desvios padrão pode enganar**: dois conjuntos de ensaios podem ter a mesma média, mas um seguir uma curva normal comportada e o outro conter um sensor saturado gerando *outliers* absurdos.

A visualização científica resolve três necessidades imediatas:
1. **Detecção de *Outliers* (Anomalias):** Picos espúrios de sensores saltam aos olhos em um gráfico em menos de 1 segundo.
2. **Inspeção de Distribuições e Tolerâncias:** Verificar se as medidas seguem um perfil gaussiano (sino) ou assimétrico.
3. **Separabilidade de Classes:** Enxergar se duas classes podem ser separadas por uma reta (Perceptron simples) ou exigem curvas (PMC).

### As Duas Bibliotecas Padrão:
- **`Matplotlib` (`plt`):** A prancheta base de desenho técnico do Python — oferece controle milimétrico de eixos, legendas, grades e subgráficos.
- **`Seaborn` (`sns`):** Construída sobre o `Matplotlib`, conversa nativamente com `DataFrames` do `Pandas` e cria gráficos estatísticos elegantes com uma única linha.

In [ ]:
import numpy as np                  # Geração de eixos numéricos e funções matemáticas
import pandas as pd                 # Leitura e manipulação da tabela CSV
import matplotlib.pyplot as plt     # Base de plotagem científica
import seaborn as sns               # Gráficos estatísticos integrados ao Pandas

# Garante que os gráficos sejam renderizados dentro do próprio caderno Jupyter
%matplotlib inline

# Aplica um tema visual limpo com grade de fundo para facilitar leitura de escalas
sns.set_theme(style="whitegrid")

## 2. Anatomia de um Gráfico Técnico no `Matplotlib`

Todo gráfico de engenharia precisa ter **grandeza e unidade explícitas nos eixos**! No `Matplotlib`, montamos o gráfico em camadas:
1. **Figura (`plt.figure`):** A folha de papel e suas dimensões `(largura, altura)` em polegadas.
2. **Curva (`plt.plot` / `plt.scatter`):** Os pontos $(x, y)$ desenhados.
3. **Anotações (`title`, `xlabel`, `ylabel`, `legend`, `grid`):** Identificação técnica obrigatória.

In [ ]:
# Simulando 200 amostras de um sinal de tensão alternada amortecida entre 0 e 2 segundos
tempo = np.linspace(0.0, 2.0, 200)                           # Eixo horizontal de tempo (s)
tensao = np.exp(-1.2 * tempo) * np.sin(2 * np.pi * 2 * tempo) # Sinal amortecido (V)

plt.figure(figsize=(9, 4))                                   # Folha de 9x4 polegadas
plt.plot(tempo, tensao, color="royalblue", linewidth=2, label="Resposta Transitória")
plt.axhline(0.0, color="black", linestyle="--", linewidth=0.8) # Linha de referência em 0 V

# Rótulos técnicos obrigatórios com unidades no SI
plt.title("Oscilação Amortecida de Circuito RLC", fontsize=13)
plt.xlabel("Tempo (s)")
plt.ylabel("Tensão (V)")
plt.legend()                                                 # Exibe a caixa de legenda
plt.grid(True, alpha=0.5)                                    # Ativa grade auxiliar suave
plt.tight_layout()                                           # Ajusta margens automaticamente
plt.show()

**Análise:** Com poucas linhas construímos um gráfico pronto para relatório técnico: título descritivo, unidades nos dois eixos, linha de zero e legenda.

---

## 3. Gráficos Estatísticos com `Seaborn` em Dados Tabulares

Vamos carregar a base do Titanic (`arquivos-csv/titanic.csv`), já tratando `'?'` como valor ausente (`na_values=['?']`), para investigar distribuições e relações entre variáveis.

In [ ]:
# Lê o CSV no caminho relativo correto e já converte '?' automaticamente em NaN
df = pd.read_csv("arquivos-csv/titanic.csv", na_values=["?"])

# Garante tipos numéricos nas colunas contínuas
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["fare"] = pd.to_numeric(df["fare"], errors="coerce")

print(f"Base carregada: {len(df)} passageiros.")

### 3.1. Histograma e Curva de Densidade (`histplot` + `kde=True`)
O **histograma** fatia uma variável contínua em faixas (*bins*) e conta quantos registros caem em cada faixa. O parâmetro `kde=True` (*Kernel Density Estimation*) desenha uma curva suave mostrando o formato da distribuição.

In [ ]:
plt.figure(figsize=(9, 4.5))

# Plota a distribuição de idades dividindo em 30 faixas (bins) com curva suave (kde=True)
sns.histplot(data=df, x="age", bins=30, kde=True, color="seagreen")

plt.title("Distribuição de Idades dos Passageiros do Titanic")
plt.xlabel("Idade (anos)")
plt.ylabel("Quantidade de Passageiros")
plt.tight_layout()
plt.show()

**Interpretação:** A maior concentração de passageiros está entre **20 e 35 anos**, com um pequeno pico secundário de crianças pequenas (`0 a 5 anos`) e cauda longa até `80 anos`.

### 3.2. Comparando Grupos e *Outliers* com `Boxplot` (Diagrama de Caixa)
O **Boxplot** resume cinco estatísticas vitais de uma vez:
- **Traço central da caixa:** A **mediana** (`50%`).
- **Bordas inferior e superior da caixa:** 1º quartil (`25%`) e 3º quartil (`75%`) — metade de todos os dados mora dentro da caixa!
- **Hastes (*bigodes*) e pontos isolados:** Limite esperado de variação e os ***outliers*** (valores extremos).

> **Nota técnica:** Nas versões atuais do `Seaborn` (`>= 0.13`), ao usar `palette` (paleta de cores), associamos `hue` à mesma variável categórica e usamos `legend=False` para não gerar avisos de depreciação (`FutureWarning`).

In [ ]:
plt.figure(figsize=(8, 5))

# Compara a distribuição de tarifas (fare) nas três classes (pclass = 1, 2, 3)
sns.boxplot(data=df, x="pclass", y="fare", hue="pclass", palette="Set2", legend=False)

# Limitamos o eixo vertical em £ 250 para enxergar claramente o corpo das caixas
plt.ylim(0, 250)
plt.title("Distribuição das Tarifas Pagas por Classe do Bilhete")
plt.xlabel("Classe Socioeconômica (1ª, 2ª e 3ª)")
plt.ylabel("Tarifa (£)")
plt.tight_layout()
plt.show()

**Interpretação:** A mediana da 1ª classe é várias vezes superior às da 2ª e 3ª classes, além de exibir diversos *outliers* acima de `£ 150` (suítes de luxo).

### 3.3. Adicionando uma Terceira Dimensão com `hue` (`countplot`)
O parâmetro `hue` (*matiz/cor*) subdivide cada barra ou nuvem de pontos por uma segunda categoria — ideal para cruzar **classe** com **sobrevivência (`0` = Não, `1` = Sim)**.

In [ ]:
plt.figure(figsize=(8, 5))

# Conta passageiros por classe (eixo x), separando por cor quem sobreviveu (hue='survived')
sns.countplot(data=df, x="pclass", hue="survived", palette="viridis")

plt.title("Contagem de Sobreviventes (1) e Não Sobreviventes (0) por Classe")
plt.xlabel("Classe do Passageiro")
plt.ylabel("Número de Passageiros")
plt.legend(title="Sobreviveu?", labels=["Não (0)", "Sim (1)"])
plt.tight_layout()
plt.show()

**Interpretação:** Na 1ª classe a barra de sobreviventes supera a de vítimas; na 3ª classe ocorre o inverso dramático.

## Guia Rápido: Qual Gráfico Escolher?

| Pergunta de Engenharia | Gráfico Ideal | Comando |
| :--- | :--- | :--- |
| Como uma grandeza evolui no tempo ou por época? | Linha | `plt.plot()` / `sns.lineplot()` |
| Onde os dados se concentram? É uma curva normal? | Histograma | `sns.histplot(..., kde=True)` |
| Existem *outliers* ou diferença de mediana entre lotes? | Boxplot | `sns.boxplot()` |
| Duas variáveis contínuas crescem juntas (correlação)? | Dispersão (*Scatter*) | `sns.scatterplot()` |
| Qual a contagem de peças aprovadas vs. defeituosas? | Contagem em Barras | `sns.countplot()` |

---

## 4. Exercícios Práticos de Fixação

### Exercício 1: Gráfico de Dispersão (`scatterplot`) — Idade vs. Tarifa
Crie um gráfico de dispersão (`sns.scatterplot`) relacionando `x='age'` e `y='fare'` no `DataFrame` `df`, colorindo os pontos por `hue='survived'` com `alpha=0.6` (transparência). Adicione título e rótulos nos eixos.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 5))
# Cada ponto representa um passageiro posicionado por idade (x) e tarifa (y)
sns.scatterplot(data=df, x="age", y="fare", hue="survived", alpha=0.6, palette="coolwarm")

plt.title("Relação entre Idade, Tarifa Paga e Sobrevivência")
plt.xlabel("Idade (anos)")
plt.ylabel("Tarifa (£)")
plt.tight_layout()
plt.show()
```

</details>

### Exercício 2: Subgráficos Lado a Lado (`plt.subplots`)
Na engenharia, frequentemente comparamos dois sinais lado a lado na mesma página usando `fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))`. No eixo `ax1`, plote `sen(x)` e no eixo `ax2` plote `cos(x)` para `x = np.linspace(0, 2*np.pi, 100)`.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import numpy as np
import matplotlib.pyplot as plt

# Malha de ângulos de 0 a 2*pi radianos
x = np.linspace(0.0, 2.0 * np.pi, 100)

# Cria 1 linha e 2 colunas de subgráficos independentes (ax1 e ax2)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x, np.sin(x), color="crimson", linewidth=2)
ax1.set_title("Função Seno: sen(x)")
ax1.set_xlabel("Ângulo (rad)")
ax1.set_ylabel("Amplitude")
ax1.grid(True)

ax2.plot(x, np.cos(x), color="navy", linewidth=2)
ax2.set_title("Função Cosseno: cos(x)")
ax2.set_xlabel("Ângulo (rad)")
ax2.set_ylabel("Amplitude")
ax2.grid(True)

plt.tight_layout()
plt.show()
```

</details>

### Exercício 3: Visualizando Curvas de Funções de Ativação Neurais
Plote no mesmo gráfico, para `u = np.linspace(-6, 6, 200)`, a curva **Logística/Sigmóide** $\sigma(u) = \frac{1}{1 + e^{-u}}$ e a curva **Tangente Hiperbólica** (`np.tanh(u)`), incluindo legenda para comparar seus intervalos de saída (`[0, 1]` vs. `[-1, 1]`).

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import numpy as np
import matplotlib.pyplot as plt

u = np.linspace(-6.0, 6.0, 200)
sigmoide = 1.0 / (1.0 + np.exp(-u))  # Faixa de saída: 0 a 1
tangente_hip = np.tanh(u)            # Faixa de saída: -1 a +1

plt.figure(figsize=(8, 4.5))
plt.plot(u, sigmoide, label="Logística / Sigmóide [0, 1]", linewidth=2.2)
plt.plot(u, tangente_hip, label="Tangente Hiperbólica [-1, 1]", linestyle="--", linewidth=2.2)
plt.axhline(0.0, color="gray", linewidth=0.8)
plt.axvline(0.0, color="gray", linewidth=0.8)

plt.title("Comparação de Funções de Ativação Clássicas")
plt.xlabel("Potencial de Ativação (u)")
plt.ylabel("Saída do Neurônio g(u)")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
```

</details>

---

## Próximo Passo

Sua caixa de ferramentas de Engenharia de Dados (`Python` + `NumPy` + `Pandas` + `Matplotlib/Seaborn`) está completa! Você pode praticá-la nos 3 exercícios guiados da pasta [`titanic/`](titanic/README.md) ou seguir direto para construir sua primeira Rede Neural Artificial no **[Módulo 05 — Teoria e Prática do Perceptron](05_perceptron_teoria_e_pratica.ipynb)**!